### **Student Information**
GitHub ID:guaisme

Kaggle name:Joyce

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# **Project Report**

---

## 1. Model Development (10 pts Required)

**Result:** Private LB **0.6693, rank 4** (Public 0.6407). This is an equal-weight probability blend of 10 fine-tuned transformer models and one TF-IDF + Logistic Regression model.
Validation used 5-fold grouped out-of-fold (OOF) macro F1: the final blend scores **0.6459**, against 0.509 for the TF-IDF baseline.

| step | OOF macro F1 | Public | Private |
|---|---|---|---|
| TF-IDF + LinearSVC baseline | 0.509 | 0.5576 | — |
| D01: 3 fine-tuned encoders + TF-IDF | 0.6207 | 0.6054 | — |
| D40: + Qwen2.5-7B QLoRA (classification head) | 0.6288 | 0.6190 | — |
| D50: + Qwen2.5-7B with verbalizer head | 0.6364 | 0.6237 | — |
| D60: + Mistral-7B, 2 emotion-pretrained encoders (9 models) | 0.6413 | 0.6489 | 0.6586 |
| **D70: + Mistral-7B verbalizer, Qwen2.5-14B verbalizer (11 models)** | **0.6459** | 0.6407 | **0.6693** |

### 1.1 Preprocessing Steps

- **Data assembly.** I joined `data_identification.csv`, `final_posts.json` and `emotion.csv` on `post_id`. This gives 3,360 training posts and 1,440 test posts. The training labels are joy 875, anger 674, sadness 628, disgust 454, surprise 441 and fear 288.
- **Audit.** No text was empty. 50 training rows were exact duplicates, and 17 groups of identical text carried *different* labels (34 rows), indicating annotation ambiguity or label inconsistency. 60 test posts also appear in the training set after normalisation (lower-case, no punctuation). I did **not** copy training labels to these matching test posts; all test predictions came from the models.
- **Grouped validation.** I normalised the text (lower-case, punctuation removed, whitespace collapsed) and used it as a group key in `StratifiedGroupKFold(5, seed 42)`. Posts with the same normalised text therefore never sit on both sides of a fold. Every model uses the same fold file (`folds_v1.csv`), so OOF predictions are comparable and can be blended.
- **Text cleaning.** I kept cleaning minimal on purpose: transformers use the raw text, including emojis, `[NAME]` and `<username>`. The only exception is the Twitter-pretrained model (G01), where I normalised username placeholders to `@user` and URLs to `http` to match its pre-training format.
- **Length.** Measured with the RoBERTa tokenizer, p95 = 43 tokens and p99 = 103 tokens, so encoders use `max_len` 128 (0.8% truncated) or 160. The LLMs use 256 tokens including the prompt. Only the post text is truncated, never the instruction.
- **Submission format.** The header is `id,label`, and every file is checked by `validate_submission.py` (1,440 rows, same order as the sample, valid labels).

### 1.2 Feature Engineering Steps

- **Sparse features (TF-IDF member).** Word 1–2-grams (keeping `!` and `?` as tokens) plus character `char_wb` 2–5-grams, with sublinear TF, feed a Logistic Regression (C = 10, balanced class weights). Adding character n-grams gave +0.03 over word n-grams alone, because they capture misspellings, elongations and emoji fragments.
- **Learned features (transformers).** The fine-tuned encoders learn their own representations through sub-word tokenisation and the `[CLS]`-based classification head. No hand-made features were added.
- **Prompt as a feature for LLMs.** Each post is wrapped in a fixed instruction:
  *"Classify the primary emotion of this social media post into one of: anger, disgust, fear, joy, sadness, surprise. Pay close attention to sarcasm, context, and subtle distinctions between anger and disgust.\nPost: {text}\nEmotion:"*
- **Verbalizer (the most useful single idea).** Instead of a new random 6-way head, the class score is the model's next-token logit for a selected token representing each label (`" anger"`, `" disgust"`, …) at the last real token. The implementation uses the first token of each space-prefixed label string, rather than scoring a full multi-token word. These are frozen rows of the pretrained LM head, so the task matches what the model already learned during pre-training. On Qwen2.5-7B this raised OOF from 0.6202 to **0.6363**, winning all 5 folds.
- **Ensemble features.** Every model outputs 6 class probabilities per post (OOF for train, averaged over the 5 fold-models for test). The final prediction is the argmax of their unweighted mean.

### 1.3 Explanation of Your Model

**Final model = equal-weight average of the class probabilities of 11 models, then argmax.**

| family | member | setting | OOF |
|---|---|---|---|
| sparse | TF-IDF + LogReg | see 1.2 | 0.495 |
| encoder | DeBERTa-v3-base (C13) | L128, lr 2e-5, 6 ep | 0.598 |
| encoder | RoBERTa-base + R-Drop (N01) | L128, lr 2e-5, 4 ep, R-Drop 1.0 | 0.604 |
| encoder | RoBERTa-large (N02) | L128, lr 1e-5, head LR ×5, 4 ep | 0.610 |
| emotion-pretrained encoder | twitter-roberta-large-emotion (G01) | L160, lr 1e-5, head ×5, 4 ep, new 6-way head | 0.620 |
| emotion-pretrained encoder | GoEmotions DeBERTa-v3-large (G02) | L160, lr 8e-6, head ×5, 4 ep | 0.624 |
| LLM, cls head | Qwen2.5-7B base (L01) | QLoRA, 2 ep | 0.620 |
| LLM, cls head | Mistral-7B-v0.3 base (M01, "MB" in the code) | QLoRA, 2 ep | 0.624 |
| LLM, verbalizer | Qwen2.5-7B base (V01) | QLoRA, 2 ep | 0.636 |
| LLM, verbalizer | Mistral-7B-v0.3 base (M02) | QLoRA, 2 ep | 0.628 |
| LLM, verbalizer | Qwen2.5-14B base (Q14) | QLoRA, 2 ep | 0.634 |
| **blend D70** | equal weights | — | **0.6459** |

- **Encoders.** Fine-tuned end-to-end with AdamW, linear warm-up and fp16 AMP. A 5× higher learning rate on the new classification head gave RoBERTa-large +0.014 (0.596 → 0.610).
- **LLMs (QLoRA).** The base model is loaded in 4-bit NF4 with fp16 compute, because the T4 does not natively support bf16. LoRA adapters (r = 16, α = 32, dropout 0.05) are trained on all attention and MLP projections for 2 epochs, with an effective batch of 16 and adapter lr 1e-4. Classification-head models also train their 6-way head; verbalizer models keep the selected LM-head rows frozen. The 14B model needed a custom low-memory loading path to fit in a 16 GB T4 (see 1.4, prompt 14).
- **Why a blend.** Single models top out at about 0.63–0.64, and their errors overlap only partly: Qwen and Mistral agree on about 82% of posts, Qwen-7B and Qwen-14B on 86.5%. Averaging the probabilities of different model families gives about +0.01 over the best single model. The best OOF-validated blend without the two emotion-pretrained encoders (D71s, 8 models) scored 0.6560 on the Private leaderboard.
- **How I limited selection overfitting.** I fixed the number of epochs in advance and scored the last epoch. The final blend used fixed equal weights. Rank averaging, logit averaging, stacking, learned weights and per-class thresholds were tested, but none beat equal weights reliably. I used +0.005 OOF as a screening guide rather than a strict cutoff, considering both the blend improvement and its consistency across folds. Repeated experiments on the same folds can still make the selected OOF score optimistic.
- **Public vs Private.** The Public split has only ~432 rows. D60 had the best Public score (0.6489), but D70, chosen on OOF evidence despite a lower Public score (0.6407), scored **+0.0107 higher on Private** (0.6693 vs 0.6586).
- **Remaining weakness.** disgust F1 is 0.479 against 0.61–0.79 for the other classes, and 120 of 454 disgust posts are predicted as anger. None of the tested methods (class weights, a disgust specialist, a probability multiplier, prompt guidance) fixed this.
- **What did not help** (all measured): class-weighted CE (−0.012), layer-wise LR decay (−0.007), BERTweet, generic DeBERTa-v3-large and ELECTRA-large (unstable on some folds), extra seeds or epochs, Qwen3-8B (85% identical predictions to Qwen2.5), a Mistral-Instruct model, and Gemini's fixed blend weights (−0.0018).
- **Disclosure.** The rules did not explicitly address emotion-fine-tuned checkpoints; after reviewing them, I interpreted the use of public pretrained weights as acceptable and chose G01 and G02 for their relevant Twitter/GoEmotions training domains. I loaded their pretrained weights and fine-tuned them using only the competition’s 3,360 training rows and the same grouped folds; no external labelled rows were added. The local folds do not establish whether these checkpoints had previously seen overlapping source texts.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

**Tools:** Claude Code (Claude Opus 5.5) for code, experiments and analysis; ChatGPT ("Astra") and Gemini as reviewers. The full log is in [competition/AI_PROMPTS.md](./competition/AI_PROMPTS.md).

| # | My prompt (short) | AI suggestion | Decision | Why |
|---|---|---|---|---|
| 1 | Audit the data and set up validation | Grouped stratified 5-fold on normalised text (duplicates, conflicting labels, shared test texts) | Adopted | keeps normalised duplicates within one fold, reducing leakage in local validation |
| 2 | Build a Kaggle baseline | Word + char TF-IDF + LinearSVC (OOF 0.509, Public 0.5576) | Adopted | provided a reference score under the same validation setup for judging later improvements |
| 3 | Which loss and max length? | Measure token lengths, use 128; compare CE vs weighted CE | Adopted 128; weighted CE rejected (−0.012) | 128 covered most posts at lower memory cost, while weighted CE reduced OOF performance |
| 7a | Review my code and validation (ChatGPT) | Scheduler stepped on AMP-skipped updates, `--full_fit` did nothing, add ID metadata, "nested CV" wording was wrong | Adopted after reviewing relevant code or test outputs | I understood the reasons for the fixes, while AI handled implementation and further checks |
| 7b | How to improve disgust? (ChatGPT) | anger/disgust specialist model, LLRD | Both rejected (specialist +0.0026, LLRD −0.007) | the specialist’s small gain did not justify the added complexity, and LLRD reduced performance |
| 7c/7d | Next steps to reach 0.65 (ChatGPT) | Classification-head LR ×5, R-Drop | Head LR ×5 adopted (+0.014); R-Drop single-model ≈ 0 | the higher head LR improved RoBERTa-large; R-Drop was assessed by its contribution to the blend rather than its standalone score |
| 8 | Add a 7B LLM via QLoRA | Qwen2.5-7B **base** (not Instruct) with QLoRA | Adopted (L01 0.620, blend +0.0036) | it added a decoder-based model and improved the blend, suggesting useful complementary predictions |
| 10 | External list: verbalizer head, Mistral, fusion tricks, multipliers | Tested the cheap claims first | **Verbalizer adopted (+0.016)**; logit/rank/z fusion and disgust multiplier rejected | the verbalizer improved all five folds, whereas the extra fusion and threshold adjustments did not help reliably |
| 12 | Use emotion-fine-tuned checkpoints | Claude warned of indirect label leakage and suggested asking the TA | Adopted after reviewing the competition rules (G01, G02) | I interpreted public pretrained weights as acceptable after reviewing the rules, and their Twitter/GoEmotions domains were relevant; fine-tuning used only the competition training rows |
| 13 | Gemini: improve Qwen-14B + verbalizer | Bigger model, multi-word verbalizer, third LLM family, fixed weights | Q14 run and added to the final blend; fixed weights rejected; multi-word and OLMo not run (time) | I prioritised testing a larger model with the already useful verbalizer; the proposed fixed weights did not improve OOF, and time limited further experiments |
| 14 | Q14 crashed with CUDA OOM | Diagnosed the PEFT fp32 up-cast of the 14B embedding/LM-head; added `--low_mem` | Adopted | made the 14B model trainable on a T4 |
| 15 | Colab keeps disconnecting | Run one fold per session, keep the tab active, write outputs to Drive | Adopted | lost one run before this |
| 16 | Final selection | Prioritise OOF evidence over the highest Public score | D70 + D60_cv selected | D70 had higher OOF and consistent gains across folds; I kept D60_cv for its stronger Public result, before the Private scores were known |

---


**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
# 1.1 Setup: paths, label order, seed. Every model below uses the same label order and the same folds.
import hashlib
import json
import math
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold

COMP_DIR = Path("data/competition")    # Kaggle competition files (download from the competition Data page)
WORK_DIR = Path("competition")         # folds, saved model probabilities, submissions
PROB_DIR = WORK_DIR / "probs"
LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
L2I = {l: i for i, l in enumerate(LABELS)}
SEED = 42
WORK_DIR.mkdir(exist_ok=True); PROB_DIR.mkdir(exist_ok=True)

In [2]:
# 1.2 Load and join the three Kaggle files on post_id; keep the test rows in sampleSubmission order.
ident = pd.read_csv(COMP_DIR / "data_identification.csv")
emo = pd.read_csv(COMP_DIR / "emotion.csv")
posts = pd.DataFrame(json.load(open(COMP_DIR / "final_posts.json", encoding="utf-8")))
sample = pd.read_csv(COMP_DIR / "sampleSubmission.csv")

df = ident.merge(posts, on="post_id", how="left").merge(emo, on="post_id", how="left")
train = df[df.set == "train"].reset_index(drop=True)
test = sample[["post_id"]].merge(df, on="post_id", how="left")

# sanity checks: unique ids, every row has text, every train row has a known label, test order = sample order
for frame in (ident, emo, posts, sample):
    assert frame.post_id.is_unique
assert len(train) == 3360 and len(test) == 1440
assert train.text.notna().all() and test.text.notna().all() and train.label.isin(LABELS).all()
assert list(test.post_id) == list(sample.post_id)
y = train.label.map(L2I).values
print("train", train.shape, "test", test.shape)
print(train.label.value_counts().to_string())

train (3360, 4) test (1440, 4)
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288


In [3]:
# 1.3 Data audit: duplicates, conflicting labels, overlap with test, length, placeholders.
def normalize(s):
    """Grouping key: lower-case, drop punctuation, collapse whitespace (used only for grouping, not as model input)."""
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", s.lower())).strip()


train["norm"] = train.text.map(normalize)
test["norm"] = test.text.map(normalize)
groups = train.groupby("norm").label
n_conflict = (groups.nunique() > 1).sum()
print("empty texts:", (train.text.str.strip() == "").sum())
print("exact duplicate train rows:", train.text.duplicated().sum())
print("groups of identical (normalized) text with different labels:", n_conflict,
      "| rows:", train.norm.isin(groups.nunique()[lambda s: s > 1].index).sum())
shared = test.norm.isin(set(train.norm))
print("test rows whose text also occurs in train (NOT used to look up labels):", shared.sum(), "| distinct texts:", test.norm[shared].nunique())
words = train.text.str.split().str.len()
print("words per post: median", words.median(), "| p95", words.quantile(.95), "| max", words.max())
print("posts with [NAME]:", train.text.str.contains(r"\[NAME\]").sum(), "| with <username>:", train.text.str.contains("<username>").sum())

empty texts: 0
exact duplicate train rows: 50
groups of identical (normalized) text with different labels: 17 | rows: 34
test rows whose text also occurs in train (NOT used to look up labels): 60 | distinct texts: 60
words per post: median 13.0 | p95 33.0 | max 508
posts with [NAME]: 236 | with <username>: 102


In [4]:
# 1.4 Leakage-safe validation: StratifiedGroupKFold grouped by the normalized text,
# so duplicates / near-duplicates never sit on both sides of a fold. All models use these exact folds.
group_ids = train.norm.factorize()[0]
train["fold"] = -1
for k, (_, va) in enumerate(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(train.text, y, group_ids)):
    train.loc[va, "fold"] = k
assert train.groupby("norm").fold.nunique().max() == 1      # no group is split across folds
fold = train.fold.values
saved = WORK_DIR / "folds_v1.csv"
if saved.exists():                                          # the file the models were trained with
    assert (train[["post_id"]].merge(pd.read_csv(saved), on="post_id").fold.values == fold).all(), "folds differ"
else:
    train[["post_id", "fold"]].to_csv(saved, index=False)
print(pd.crosstab(train.fold, train.label))

label  anger  disgust  fear  joy  sadness  surprise
fold                                               
0        135       91    58  175      125        88
1        135       91    57  175      126        88
2        135       91    57  175      126        88
3        135       91    58  175      125        88
4        134       90    58  175      126        89


In [5]:
# 1.5 Text cleaning. Models read the raw text (emojis, [NAME], <username> kept). The only rewrite is for the
# Twitter-pretrained encoder (member G01), whose pre-training data used "@user" and "http" placeholders.
def twitter_norm(s):
    s = re.sub(r"@?<username>", "@user", s)
    s = re.sub(r"(?<![\w@])@\w+", "@user", s)
    return re.sub(r"##URL##|https?://\S+|www\.\S+", "http", s)


print(next(t for t in train.text if "<username>" in t)[:80], "->", twitter_norm(next(t for t in train.text if "<username>" in t))[:80])

@<username> I feel sick #Famille -> @user I feel sick #Famille


## 2. Feature Engineering Steps

In [6]:
# 2.1 Sparse features for the TF-IDF member: word 1-2-grams (keeping ! and ?) + character 2-5-grams (char_wb),
# sublinear TF, Logistic Regression. The vectorizers are fit inside each training fold only (Pipeline).
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import FeatureUnion, Pipeline


def tfidf_lr():
    feats = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b|[!?]")),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True))])
    return Pipeline([("f", feats), ("m", LogisticRegression(C=10, class_weight="balanced", max_iter=3000))])


tf_oof, tf_test = np.zeros((len(train), 6)), np.zeros((len(test), 6))
for k in range(5):
    tr, va = fold != k, fold == k
    m = tfidf_lr().fit(train.text[tr], y[tr])
    tf_oof[va] = m.predict_proba(train.text[va])
    tf_test += m.predict_proba(test.text) / 5          # test = mean of the 5 fold models
print("TF-IDF + LR OOF macro F1:", round(f1_score(y, tf_oof.argmax(1), average="macro"), 4))
if (PROB_DIR / "oof_TF.npy").exists():   # same as the saved member used in the blend?
    print("max |difference| vs saved TF member:", float(np.abs(tf_oof - np.load(PROB_DIR / "oof_TF.npy")).max()))
else:
    np.save(PROB_DIR / "oof_TF.npy", tf_oof); np.save(PROB_DIR / "test_TF.npy", tf_test)

TF-IDF + LR OOF macro F1: 0.4953
max |difference| vs saved TF member: 0.0


In [7]:
# 2.2 Prompt + verbalizer for the LLM members. The post is wrapped in a fixed instruction; only the post text is
# truncated (never the instruction). The verbalizer scores each class by the LLM's own next-token logit of the label
# word (" anger", " disgust", ...) after "Emotion:", using frozen rows of the pretrained output layer.
PREFIX = ("Classify the primary emotion of this social media post into one of: anger, disgust, fear, joy, "
          "sadness, surprise. Pay close attention to sarcasm, context, and subtle distinctions between anger "
          "and disgust.\nPost: ")
SUFFIX = "\nEmotion:"


def encode_prompts(tok, texts, max_len=256):
    """token ids = PREFIX + text (truncated to fit) + SUFFIX"""
    pre = tok(PREFIX, add_special_tokens=False)["input_ids"]; suf = tok(SUFFIX, add_special_tokens=False)["input_ids"]
    budget = max_len - len(pre) - len(suf)
    return [pre + t[:budget] + suf for t in tok(list(texts), add_special_tokens=False)["input_ids"]]


def label_token_ids(tok):
    """first token of ' <label>' (leading space, as it follows 'Emotion:'); must be different for all six labels"""
    ids = [tok(" " + l, add_special_tokens=False)["input_ids"][0] for l in LABELS]
    assert len(set(ids)) == 6, ids
    return ids


try:  # small download (tokenizer only) to show what the LLM members see
    from transformers import AutoTokenizer
    qtok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")
    seqs = encode_prompts(qtok, train.text)
    print("prompt example:\n" + qtok.decode(seqs[0]))
    print("label tokens:", [qtok.decode([t]) for t in label_token_ids(qtok)])
    print("posts truncated at 256 tokens:", sum(len(s) >= 256 for s in seqs))
except Exception as e:
    print("skipped (transformers or the tokenizer download not available):", type(e).__name__)

prompt example:
Classify the primary emotion of this social media post into one of: anger, disgust, fear, joy, sadness, surprise. Pay close attention to sarcasm, context, and subtle distinctions between anger and disgust.
Post: Thought it looked weird as well.
Emotion:
label tokens: [' anger', ' disgust', ' fear', ' joy', ' sadness', ' surprise']
posts truncated at 256 tokens: 18


## 3. Model Implementation Steps

In [8]:
# 3.1 The 11 members of the final blend (D70) and their fixed settings. Epochs were fixed in advance and the last
# epoch is scored. "enc" = encoder fine-tuning (3.2), "llm" = 4-bit QLoRA on a decoder LLM (3.3).
MEMBERS = {
    "TF":  dict(kind="tfidf"),                                                                  # 2.1
    "C13": dict(kind="enc", model="microsoft/deberta-v3-base", max_len=128, lr=2e-5, epochs=6, bs=16),
    "N01": dict(kind="enc", model="roberta-base", max_len=128, lr=2e-5, epochs=4, bs=16, rdrop=1.0),
    "N02": dict(kind="enc", model="roberta-large", max_len=128, lr=1e-5, epochs=4, bs=16, head_lr_mult=5, seed=7),
    "G01": dict(kind="enc", model="cardiffnlp/twitter-roberta-large-emotion-latest", max_len=160, lr=1e-5, epochs=4,
                bs=16, head_lr_mult=5, twitter=True),
    "G02": dict(kind="enc", model="duelker/samo-goemotions-deberta-v3-large", max_len=160, lr=8e-6, epochs=4,
                bs=4, grad_accum=4, head_lr_mult=5, grad_ckpt=True),
    "L01": dict(kind="llm", model="Qwen/Qwen2.5-7B", head="cls"),
    "MB":  dict(kind="llm", model="mistralai/Mistral-7B-v0.3", head="cls"),
    "V01": dict(kind="llm", model="Qwen/Qwen2.5-7B", head="verbalizer"),
    "M02": dict(kind="llm", model="mistralai/Mistral-7B-v0.3", head="verbalizer"),
    "Q14": dict(kind="llm", model="Qwen/Qwen2.5-14B", head="verbalizer", bs=2, grad_accum=8, low_mem=True),
}
RUN_TRAINING = False   # True only on a GPU (T4 16 GB): about 0.1-2 h per member per fold
SMOKE_TEST = False     # True: run 3.2/3.3 on tiny random models for a few steps on CPU, to check the code path


def seed_all(s):
    random.seed(s); np.random.seed(s)
    import torch
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [9]:
# 3.2 Encoder fine-tuning (DeBERTa / RoBERTa family): pretrained encoder + new 6-way classification head,
# AdamW with linear warm-up/decay, fp16 autocast, optional 5x LR on the new head, optional R-Drop
# (two dropout passes + symmetric KL), gradient accumulation and gradient checkpointing for the large models.
def train_encoder_cv(cfg, max_steps=0, folds=range(5)):
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup
    c = dict(lr=2e-5, epochs=4, bs=16, grad_accum=1, max_len=128, head_lr_mult=1.0, rdrop=0.0, wd=0.01, warmup=0.1,
             seed=SEED, twitter=False, grad_ckpt=False) | cfg
    dev = "cuda" if torch.cuda.is_available() else "cpu"; amp = dev == "cuda"
    tok = AutoTokenizer.from_pretrained(c["model"])
    prep = twitter_norm if c["twitter"] else (lambda s: s)
    tr_text = [prep(s) for s in train.text]; te_text = [prep(s) for s in test.text]

    def loader(texts, labels=None, shuffle=False):
        enc = tok(list(texts), truncation=True, max_length=c["max_len"])
        items = [{**{k: v[i] for k, v in enc.items()}, **({"labels": int(labels[i])} if labels is not None else {})}
                 for i in range(len(texts))]

        def collate(b):
            out = tok.pad([{k: v for k, v in x.items() if k != "labels"} for x in b], return_tensors="pt")
            if "labels" in b[0]:
                out["labels"] = torch.tensor([x["labels"] for x in b])
            return out
        g = torch.Generator(); g.manual_seed(c["seed"])
        return DataLoader(items, batch_size=c["bs"] if shuffle else 32, shuffle=shuffle, collate_fn=collate, generator=g)

    @torch.no_grad()
    def predict(model, dl):
        model.eval(); out = []
        for b in dl:
            b = {k: v.to(dev) for k, v in b.items() if k != "labels"}
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                out.append(torch.softmax(model(**b).logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    def sym_kl(l1, l2):
        p, q = torch.log_softmax(l1.float(), -1), torch.log_softmax(l2.float(), -1)
        return 0.5 * ((p.exp() * (p - q)).sum(-1).mean() + (q.exp() * (q - p)).sum(-1).mean())

    oof, test_p = np.full((len(train), 6), np.nan), []
    te_dl = loader(te_text)
    for k in folds:
        tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
        seed_all(c["seed"])
        model = AutoModelForSequenceClassification.from_pretrained(
            c["model"], num_labels=6, ignore_mismatched_sizes=True, problem_type="single_label_classification")
        if c["grad_ckpt"]:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model = model.float().to(dev)                     # fp32 master weights; autocast runs the forward in fp16
        backbone = {id(p) for p in model.base_model.parameters()}
        groups_ = {}
        for n, p in model.named_parameters():             # new head gets lr x head_lr_mult; no decay on bias/norm
            lr = c["lr"] * (1.0 if id(p) in backbone else c["head_lr_mult"])
            wd = 0.0 if "bias" in n or "norm" in n.lower() else c["wd"]
            groups_.setdefault((lr, wd), []).append(p)
        opt = torch.optim.AdamW([{"params": v, "lr": lr, "weight_decay": wd} for (lr, wd), v in groups_.items()])
        dl = loader([tr_text[i] for i in tr_idx], y[tr_idx], shuffle=True)
        n_micro = len(dl) if not max_steps else min(len(dl), max_steps * c["grad_accum"])
        steps = math.ceil(n_micro / c["grad_accum"]) * c["epochs"]
        sch = get_linear_schedule_with_warmup(opt, int(c["warmup"] * steps), steps)
        scaler = torch.amp.GradScaler(enabled=amp)
        for ep in range(c["epochs"]):
            model.train(); opt.zero_grad(set_to_none=True)
            for i, b in enumerate(dl):
                if i >= n_micro:
                    break
                b = {kk: v.to(dev) for kk, v in b.items()}; lab = b.pop("labels")
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                    logits = model(**b).logits
                loss = torch.nn.functional.cross_entropy(logits.float(), lab)
                if c["rdrop"] > 0:                       # R-Drop: second pass with different dropout
                    with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                        logits2 = model(**b).logits
                    loss = 0.5 * (loss + torch.nn.functional.cross_entropy(logits2.float(), lab)) + c["rdrop"] * sym_kl(logits, logits2)
                group = min(c["grad_accum"], n_micro - (i // c["grad_accum"]) * c["grad_accum"])
                scaler.scale(loss / group).backward()
                if (i + 1) % c["grad_accum"] == 0 or i + 1 == n_micro:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    before = scaler.get_scale(); scaler.step(opt); scaler.update()
                    if scaler.get_scale() >= before:     # do not advance the schedule on AMP-skipped steps
                        sch.step()
                    opt.zero_grad(set_to_none=True)
        oof[va_idx] = predict(model, loader([tr_text[i] for i in va_idx]))     # last epoch is scored
        test_p.append(predict(model, te_dl))
        print(f"  fold {k}: macro F1 {f1_score(y[va_idx], oof[va_idx].argmax(1), average='macro'):.4f}")
        del model
    return oof, np.mean(test_p, 0)

In [10]:
# 3.3 QLoRA on a decoder LLM (Qwen2.5-7B/14B, Mistral-7B): base model in 4-bit NF4 with fp16 compute (T4 has no bf16),
# LoRA r=16 / alpha=32 / dropout 0.05 on all attention + MLP projections, 2 epochs, effective batch 16, cosine schedule.
# head="cls": new 6-way score head (lr 5e-4).  head="verbalizer": class logits = hidden state of the last prompt token
# x frozen output-layer rows of the 6 label words (only LoRA trains).  low_mem=True (14B on a 16 GB T4): keep the big
# embedding/output matrices in fp16 instead of PEFT's fp32 up-cast, and drop the unused full-vocabulary output layer.
def train_llm_cv(cfg, max_steps=0, folds=range(5), use_4bit=True):
    import torch
    from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer, get_cosine_schedule_with_warmup
    c = dict(max_len=256, lr=1e-4, head_lr=5e-4, epochs=2, bs=4, grad_accum=4, warmup=0.05, r=16, alpha=32, dropout=0.05,
             seed=SEED, low_mem=False) | cfg
    dev = "cuda" if torch.cuda.is_available() else "cpu"; amp = dev == "cuda"
    tok = AutoTokenizer.from_pretrained(c["model"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tr_seqs, te_seqs = encode_prompts(tok, train.text, c["max_len"]), encode_prompts(tok, test.text, c["max_len"])
    verbal = c["head"] == "verbalizer"
    lab_ids = label_token_ids(tok) if verbal else None

    def batches(seqs, labels, bs, shuffle, seed=0):
        order = np.random.default_rng(seed).permutation(len(seqs)) if shuffle else np.argsort([len(s) for s in seqs], kind="stable")
        for i in range(0, len(order), bs):
            idx = order[i:i + bs]; L = max(len(seqs[j]) for j in idx)
            ids = torch.full((len(idx), L), tok.pad_token_id); att = torch.zeros((len(idx), L), dtype=torch.long)
            for r, j in enumerate(idx):                    # right padding
                ids[r, :len(seqs[j])] = torch.tensor(seqs[j]); att[r, :len(seqs[j])] = 1
            yield idx, ids, att, (torch.tensor(labels[idx]) if labels is not None else None)

    def build():
        kw = {} if verbal else {"num_labels": 6}
        Auto = AutoModelForCausalLM if verbal else AutoModelForSequenceClassification
        if use_4bit:
            from transformers import BitsAndBytesConfig
            kw |= dict(device_map={"": 0}, quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))
        model = Auto.from_pretrained(c["model"], dtype=torch.float16 if use_4bit else torch.float32, **kw)
        model.config.pad_token_id = tok.pad_token_id
        if use_4bit and not c["low_mem"]:
            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                                    gradient_checkpointing_kwargs={"use_reentrant": False})
        elif c["low_mem"]:
            for p in model.parameters():                   # frozen; only small 1-D params (norms, biases) go to fp32
                p.requires_grad_(False)
                if p.ndim == 1 and p.dtype == torch.float16:
                    p.data = p.data.float()
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            model.enable_input_require_grads()
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.CAUSAL_LM if verbal else TaskType.SEQ_CLS, r=c["r"], lora_alpha=c["alpha"], lora_dropout=c["dropout"],
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], bias="none"))
        for p in model.parameters():                       # GradScaler needs fp32 trainable parameters
            if p.requires_grad and p.dtype != torch.float32:
                p.data = p.data.float()
        if not use_4bit:
            model = model.to(dev)
        if verbal:
            causal = model.get_base_model()
            model._label_w = causal.get_output_embeddings().weight[lab_ids].detach().float().clone()
            if c["low_mem"]:
                causal.lm_head = torch.nn.Identity()
            model._decoder = causal.get_decoder()
        return model

    def class_logits(model, ids, att):
        if not verbal:
            return model(input_ids=ids, attention_mask=att).logits
        h = model._decoder(input_ids=ids, attention_mask=att).last_hidden_state
        h_last = h[torch.arange(h.size(0), device=h.device), att.sum(1) - 1]     # last real token ("Emotion:")
        return h_last.float() @ model._label_w.T

    @torch.no_grad()
    def predict(model, seqs):
        model.eval(); out = np.zeros((len(seqs), 6), dtype=np.float32)
        for idx, ids, att, _ in batches(seqs, None, c["bs"] * 2, shuffle=False):
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                out[idx] = torch.softmax(class_logits(model, ids.to(dev), att.to(dev)).float(), -1).cpu().numpy()
        return out

    oof, test_p = np.full((len(train), 6), np.nan), []
    for k in folds:
        tr_idx, va_idx = np.where(fold != k)[0], np.where(fold == k)[0]
        seed_all(c["seed"]); model = build()
        lora = [p for n, p in model.named_parameters() if p.requires_grad and "score" not in n]
        head = [p for n, p in model.named_parameters() if p.requires_grad and "score" in n]
        opt = torch.optim.AdamW([{"params": lora, "lr": c["lr"], "weight_decay": 0.0}]
                                + ([{"params": head, "lr": c["head_lr"], "weight_decay": 0.0}] if head else []))
        per_epoch = math.ceil(math.ceil(len(tr_idx) / c["bs"]) / c["grad_accum"])
        total = min(max_steps, per_epoch * c["epochs"]) if max_steps else per_epoch * c["epochs"]
        sch = get_cosine_schedule_with_warmup(opt, int(c["warmup"] * total), total)
        scaler = torch.amp.GradScaler(enabled=amp); done = 0
        for ep in range(c["epochs"]):
            model.train(); opt.zero_grad(); micro = 0
            for idx, ids, att, lab in batches([tr_seqs[i] for i in tr_idx], y[tr_idx], c["bs"], True, c["seed"] * 100 + ep):
                with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
                    logits = class_logits(model, ids.to(dev), att.to(dev))
                scaler.scale(torch.nn.functional.cross_entropy(logits.float(), lab.to(dev)) / c["grad_accum"]).backward()
                micro += 1
                if micro % c["grad_accum"] == 0:
                    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(lora + head, 1.0)
                    before = scaler.get_scale(); scaler.step(opt); scaler.update()
                    if scaler.get_scale() >= before:
                        sch.step()
                    opt.zero_grad(); done += 1
                    if max_steps and done >= max_steps:
                        break
            if max_steps and done >= max_steps:
                break
        oof[va_idx] = predict(model, [tr_seqs[i] for i in va_idx])
        test_p.append(predict(model, te_seqs))
        print(f"  fold {k}: macro F1 {f1_score(y[va_idx], oof[va_idx].argmax(1), average='macro'):.4f}")
        del model
        if dev == "cuda":
            torch.cuda.empty_cache()
    return oof, np.mean(test_p, 0)

In [11]:
# 3.4 Train the members (GPU) and save their OOF/test probabilities. The saved files in competition/probs
# are the outputs of these runs (on Kaggle/Colab T4 GPUs, same folds), so the blend in 3.5 runs without a GPU.
if SMOKE_TEST:   # code-path check only: tiny random models, fold 0, 3 optimizer steps, scores are meaningless
    train_encoder_cv(dict(model="ydshieh/tiny-random-DebertaV2ForMaskedLM", epochs=1, rdrop=1.0,
                          head_lr_mult=5, twitter=True), max_steps=3, folds=[0])
    train_llm_cv(dict(model="peft-internal-testing/tiny-dummy-qwen2", head="verbalizer", epochs=1, low_mem=True),
                 max_steps=3, folds=[0], use_4bit=False)
    train_llm_cv(dict(model="peft-internal-testing/tiny-dummy-qwen2", head="cls", epochs=1), max_steps=3, folds=[0], use_4bit=False)
if RUN_TRAINING:
    for name, cfg in MEMBERS.items():
        if cfg["kind"] == "tfidf" or (PROB_DIR / f"oof_{name}.npy").exists():
            continue
        print("training", name, cfg)
        cfg = {k: v for k, v in cfg.items() if k != "kind"}
        o, t = (train_encoder_cv if MEMBERS[name]["kind"] == "enc" else train_llm_cv)(cfg)
        np.save(PROB_DIR / f"oof_{name}.npy", o); np.save(PROB_DIR / f"test_{name}.npy", t)

In [12]:
# 3.5 Final model = equal-weight mean of the 11 members' class probabilities, then argmax (no tuned weights).
# Single-model and blend OOF macro F1, per fold and per class; then write the submission (header id,label).
manifest = json.load(open(PROB_DIR / "MANIFEST.json"))["members"]
O, T = {}, {}
for name in MEMBERS:
    for kind, store in (("oof", O), ("test", T)):
        f = PROB_DIR / f"{kind}_{name}.npy"
        assert hashlib.sha256(f.read_bytes()).hexdigest() == manifest[name][kind], f"{f} is not the file used in the blend"
        store[name] = np.load(f)


def macro(p, rows=slice(None)):
    return f1_score(y[rows], p[rows].argmax(1), average="macro")


def per_fold(p):
    return [round(macro(p, fold == k), 4) for k in range(5)]


print(f"{'member':6s} OOF     per fold")
for name in MEMBERS:
    print(f"{name:6s} {macro(O[name]):.4f}  {per_fold(O[name])}")
blend_oof = np.mean([O[n] for n in MEMBERS], 0)
blend_test = np.mean([T[n] for n in MEMBERS], 0)
print(f"\nD70 blend OOF macro F1 {macro(blend_oof):.4f} per fold {per_fold(blend_oof)}")
print("per-class F1:", {l: round(float(v), 3) for l, v in zip(LABELS, f1_score(y, blend_oof.argmax(1), average=None))})
print(pd.DataFrame(confusion_matrix(y, blend_oof.argmax(1)), index=[f"true {l}" for l in LABELS], columns=LABELS))

submission = pd.DataFrame({"id": sample.post_id, "label": [LABELS[i] for i in blend_test.argmax(1)]})
submission.to_csv(WORK_DIR / "submission_D70_reproduced.csv", index=False)
submitted = WORK_DIR / "submission_D70_all11_equal.csv"
if submitted.exists():
    print("identical to the submitted file (Private 0.6693):", submission.label.tolist() == pd.read_csv(submitted).label.tolist())

member OOF     per fold
TF     0.4953  [0.5278, 0.5043, 0.5035, 0.4706, 0.4685]
C13    0.5976  [0.6002, 0.6181, 0.6001, 0.5698, 0.5976]
N01    0.6035  [0.606, 0.6211, 0.5971, 0.5955, 0.5954]
N02    0.6100  [0.6155, 0.6124, 0.5974, 0.6108, 0.6122]
G01    0.6204  [0.6254, 0.6485, 0.588, 0.6147, 0.6242]
G02    0.6238  [0.6131, 0.6376, 0.6256, 0.6249, 0.6189]
L01    0.6202  [0.6304, 0.6177, 0.6193, 0.608, 0.6262]
MB     0.6239  [0.6307, 0.6457, 0.619, 0.6101, 0.6137]
V01    0.6363  [0.6453, 0.6392, 0.6484, 0.616, 0.6322]
M02    0.6280  [0.6365, 0.6505, 0.6176, 0.6182, 0.6155]
Q14    0.6343  [0.6438, 0.6473, 0.6413, 0.6216, 0.6161]

D70 blend OOF macro F1 0.6459 per fold [0.6606, 0.6503, 0.6267, 0.6426, 0.6495]
per-class F1: {'anger': 0.638, 'disgust': 0.479, 'fear': 0.675, 'joy': 0.792, 'sadness': 0.681, 'surprise': 0.61}
               anger  disgust  fear  joy  sadness  surprise
true anger       444       62    18   52       64        34
true disgust     120      195    16   40       47 

In [13]:
# 3.6 How members were admitted: a candidate blend replaced the current one only if it gained >= +0.005 OOF macro F1
# AND won >= 4 of 5 folds (rule fixed before each run). A group bootstrap (resampling whole duplicate groups) gives a
# 95% interval for the gain. Example: D70 vs D60 (D60 = the 9 members without the two last LLMs M02, Q14).
def compare(cand, base, reps=2000):
    rng = np.random.default_rng(0); ug = np.unique(group_ids); idx = {g: np.where(group_ids == g)[0] for g in ug}; d = []
    for _ in range(reps):
        rr = np.concatenate([idx[g] for g in rng.choice(ug, len(ug))])
        d.append(macro(cand, rr) - macro(base, rr))
    won = sum(a > b for a, b in zip(per_fold(cand), per_fold(base)))
    return round(macro(cand) - macro(base), 4), won, np.round(np.percentile(d, [2.5, 97.5]), 4)


d60 = np.mean([O[n] for n in MEMBERS if n not in ("M02", "Q14")], 0)
gain, won, ci = compare(blend_oof, d60)
print(f"D60 {macro(d60):.4f} -> D70 {macro(blend_oof):.4f}: gain {gain:+.4f}, folds won {won}/5, 95% CI {ci}")

D60 0.6413 -> D70 0.6459: gain +0.0047, folds won 5/5, 95% CI [-0.0001  0.0095]
